<a href="https://colab.research.google.com/github/Shreyas140304/Sentiment-Analyzer/blob/main/Random_forest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**API for extracting the youtube comments**

In [ ]:
# for performing request
from googleapiclient.discovery import build

# for working on dataframe
import pandas as pd

In [ ]:
# This is our google api key, through which we will request the youtube api and extract the required data.
api_key = ''

In [ ]:
api_service_name = "youtube" # Our api name
api_version = "v3" # Version name

# Get credentials and create an API client
youtube = build(
   api_service_name, api_version, developerKey=api_key)

In [ ]:
# we have created a function to get all the video id's from the respective playlist id.
playlist_id ="PLeo1K3hjS3uuvuAXhYjV2lMEShq2UYSwX"
def get_video_ids(youtube, playlist_id):

    video_ids = []

    request = youtube.playlistItems().list(
        part="contentDetails",
        playlistId=playlist_id,
        maxResults = 50
    )
    response = request.execute()

    for item in response['items']:
        video_ids.append(item['contentDetails']['videoId'])

    next_page_token = response.get('nextPageToken')
    while next_page_token is not None:
        request = youtube.playlistItems().list(
                    part='contentDetails',
                    playlistId =playlist_id ,
                    maxResults = 50,
                    pageToken = next_page_token)
        response = request.execute()

# The for loop will iterate from the response dictionaries all keys, and the values will be appended in the video_ids list.
        for item in response['items']:
            video_ids.append(item['contentDetails']['videoId'])

        next_page_token = response.get('nextPageToken')

    return video_ids

In [ ]:

box = [['Name', 'Comment', 'Time', 'Likes', 'Reply Count']]


def scrape_comments_with_replies():
    data = youtube.commentThreads().list(part='snippet', videoId=ID, maxResults='100', textFormat="plainText").execute()

    for i in data["items"]:

        name = i["snippet"]['topLevelComment']["snippet"]["authorDisplayName"]
        comment = i["snippet"]['topLevelComment']["snippet"]["textDisplay"]
        published_at = i["snippet"]['topLevelComment']["snippet"]['publishedAt']
        likes = i["snippet"]['topLevelComment']["snippet"]['likeCount']
        replies = i["snippet"]['totalReplyCount']

        box.append([name, comment, published_at, likes, replies])

        totalReplyCount = i["snippet"]['totalReplyCount']

        if totalReplyCount > 0:

            parent = i["snippet"]['topLevelComment']["id"]

            data2 = youtube.comments().list(part='snippet', maxResults='100', parentId=parent,
                                            textFormat="plainText").execute() # Here maxResults is specified to 100, by default it is 50 and the maxResults value is 1000 max.

            for i in data2["items"]:
                name = i["snippet"]["authorDisplayName"]
                comment = i["snippet"]["textDisplay"]
                published_at = i["snippet"]['publishedAt']
                likes = i["snippet"]['likeCount']
                replies = ""

                box.append([name, comment, published_at, likes, replies])

# If there are more comments to be fetched, this while loop will run if there are more tokens to be fetched.
    while ("nextPageToken" in data):

        data = youtube.commentThreads().list(part='snippet', videoId=ID, pageToken=data["nextPageToken"],
                                             maxResults='100', textFormat="plainText").execute()

        for i in data["items"]:
            name = i["snippet"]['topLevelComment']["snippet"]["authorDisplayName"]
            comment = i["snippet"]['topLevelComment']["snippet"]["textDisplay"]
            published_at = i["snippet"]['topLevelComment']["snippet"]['publishedAt']
            likes = i["snippet"]['topLevelComment']["snippet"]['likeCount']
            replies = i["snippet"]['totalReplyCount']

            box.append([name, comment, published_at, likes, replies])

            totalReplyCount = i["snippet"]['totalReplyCount']
            print(totalReplyCount)

            if totalReplyCount > 0:

                parent = i["snippet"]['topLevelComment']["id"]

                data2 = youtube.comments().list(part='snippet', maxResults='100', parentId=parent,
                                                textFormat="plainText").execute()

                for i in data2["items"]:
                    name = i["snippet"]["authorDisplayName"]
                    comment = i["snippet"]["textDisplay"]
                    published_at = i["snippet"]['publishedAt']
                    likes = i["snippet"]['likeCount']
                    replies = ''

                    box.append([name, comment, published_at, likes, replies])

    df = pd.DataFrame({'Name': [i[0] for i in box], 'Comment': [i[1] for i in box], 'Time': [i[2] for i in box],
                       'Likes': [i[3] for i in box], 'Reply Count': [i[4] for i in box]})

    df.to_csv('testyoutube-comments.csv', index=False, header=False)

    return df

In [ ]:
video_ids = get_video_ids(youtube, playlist_id)
video_ids

['R-AG4-qZs1A',
 '3y2-IaBeIs0',
 'lK9gx4q_vfI',
 'nknYY32RGXQ',
 'In7jB8TUGPA',
 'S3EId9uatxI',
 'h2kBNEShsiE',
 '_lR3RjvYvF4',
 'hKK59rfpXL0',
 'HHAilAC3cXw',
 'gdHWoQWZGkk',
 '2XUhKpH0p4M',
 '_3ahmI5vpKY',
 '2d8iP2_cS-U',
 'Yt1Sw6yWjlw',
 'vUPAOU2NPls',
 'nZromH6F7R0',
 'ATK6fm3cYfI',
 'Do8cVbx-HOs',
 'vyohzuTkty8',
 'ibi5hvw6f3g',
 '0r2NJdalzDw',
 'ZrgVlfNduj8',
 'Br-Ozg9D4mc',
 'Cq_pbQYO3M8',
 'ZeoqOybAzdc',
 '2e5pQqBvGco',
 'Ji3_VX80YJg']

In [ ]:
print(type(video_ids))

<class 'list'>


In [ ]:
#i =int(input("Which video no. comments yo want to fetch ?"))
for i in range(len(video_ids)):
  ID = video_ids[i]
  df = scrape_comments_with_replies()

0
0
0
0
0
1
2
0
0
0
0
0
0
0
1
0
0
0
2
1
0
0
0
0
0
0
0
0
0
0
0
2
1
0
0
0
0
0
0
0
1
0
0
6
0
0
3
0
2
0
0
1
0
1
1
0
2
2
0
0
0
0
0
0
0
0
0
0
0
0
0
1
0
0
0
0
0
1
16
0
8
1
0
0
8
0
0
0
0
4
0
2
0
0
0
0
1
0
0
0
0
2
1
0
0
0
0
0
0
0
0
0
0
0
1
0
0
0
1
3
0
1
0
0
0
1
0
2
3
3
0
0
1
2
5
0
0
0
0
2
0
1
3
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
1
0
0
0
0
0
0
1
1
0
0
0
4
38
0
0
0
0
0
0
0
2
3
0
1
1
2
2
0
0
0
0
0
0
0
0
1
0
0
2
1
0
0
0
0
0
1
12
21
2
1
8
6
0
1
1
2
2
4
2
0
2
2
1
0
0
0
0
1
0
2
0
1
3
0
1
0
0
2
1
1
4


In [ ]:
import pandas as pd
import re
import nltk
nltk.download('vader_lexicon')
from nltk.sentiment.vader import SentimentIntensityAnalyzer

[nltk_data] Downloading package vader_lexicon to /root/nltk_data...


In [ ]:
cm = pd.read_csv('/content/codebasicsnlpallcomments.csv')
#df.columns = df.columns.str.replace(' ', '')
cm.columns = cm.columns.str.replace(' ', '')

In [ ]:
# Segregating comments
comments = cm.loc[cm['Name'] != "codebasics", 'Comment'].iloc[:len(cm['Name'])]
comments = pd.DataFrame(comments)

In [ ]:
# Getting the polarity scores and the labels
# We will be using the SentimentIntensityAnalyzer from nltk library for getting the polarity scores.
sia = SentimentIntensityAnalyzer()
comments['Sentiment Scores'] = comments['Comment'].apply(lambda x:sia.polarity_scores(x)['compound'])
comments['Sentiment'] = comments['Sentiment Scores'].apply(lambda s : 'Positive' if s > 0 else ('Neutral' if s == 0 else 'Negative'))

In [ ]:
# lets load our labelled comments dataset.
lb = pd.DataFrame(comments)

# Now assigning the integer values for the target variables.
lb['Sentiment'] = lb['Sentiment'].replace(to_replace='Positive', value=1)
lb['Sentiment'] = lb['Sentiment'].replace(to_replace='Neutral', value=0)
lb['Sentiment'] = lb['Sentiment'].replace(to_replace='Negative', value=-1)

<ipython-input-16-09cdfd464138>:7: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  lb['Sentiment'] = lb['Sentiment'].replace(to_replace='Negative', value=-1)


In [ ]:
X = lb['Comment']
y = lb['Sentiment']

**Train_test_split-1**

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,random_state=2022,
    stratify=y)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report
#1. create a pipeline object
clf = Pipeline([
     ('vectorizer_tfidf',TfidfVectorizer()),
     ('Random Forest', RandomForestClassifier())
])

#2. fit with X_train and y_train
clf.fit(X_train, y_train)


#3. get the predictions for X_test and store it in y_pred
y_pred = clf.predict(X_test)

#4. print the confusion matrix and classfication report.
from sklearn.metrics import classification_report, confusion_matrix
print(confusion_matrix(y_test, y_pred))

print()
print(classification_report(y_test, y_pred))

[[  2   1  11]
 [  0  34  13]
 [  1  11 120]]

              precision    recall  f1-score   support

          -1       0.67      0.14      0.24        14
           0       0.74      0.72      0.73        47
           1       0.83      0.91      0.87       132

    accuracy                           0.81       193
   macro avg       0.75      0.59      0.61       193
weighted avg       0.80      0.81      0.79       193



In [ ]:
a1=["I like your videos ","which playlist i should follow to get undrestand and build nlp model step by step . i am completely beginner in AI","in any you tube video  no man can any videos related to document based cahtbot.....can u make video document base chatbot using lstm,......when u search chatbot everyone focus on intents ,patterns ,response based chatbot predefinedrasa nlu....no make make document based chatbot.........can u make it.........?","Apart from NLP,. Time Series  tutorial with python could also help people like me that are interested in it. Your courses of Machine learning and deep Learning shaped my understanding. Thank you for your support. Best wishes from Nigeria"]

a=clf.predict(a1).tolist()
print(len(a))
for i in range(len(a)):
  if a[i] == 1:
    print("It is a positive comment \U0001f600")
  elif a[i] == 0:
    print("It is a neutral comment \U0001F610")
  else:
    print("It is a negative comment \U0001F614")

4
It is a positive comment 😀
It is a neutral comment 😐
It is a negative comment 😔
It is a positive comment 😀


**Train_test_split-2**

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25,random_state=2022,
    stratify=y)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report
#1. create a pipeline object
clf = Pipeline([
     ('vectorizer_tfidf',TfidfVectorizer()),
     ('Random Forest', RandomForestClassifier())
])

#2. fit with X_train and y_train
clf.fit(X_train, y_train)


#3. get the predictions for X_test and store it in y_pred
y_pred = clf.predict(X_test)


#4. print the confusion matrix and classfication report.
from sklearn.metrics import classification_report, confusion_matrix
print(confusion_matrix(y_test, y_pred))

print()
print(classification_report(y_test, y_pred))

[[  1   3  14]
 [  0  38  20]
 [  0  15 150]]

              precision    recall  f1-score   support

          -1       1.00      0.06      0.11        18
           0       0.68      0.66      0.67        58
           1       0.82      0.91      0.86       165

    accuracy                           0.78       241
   macro avg       0.83      0.54      0.54       241
weighted avg       0.80      0.78      0.76       241



In [ ]:
a1=["video quality is bad","which playlist i should follow to get undrestand and build nlp model step by step . i am completely beginner in AI","in any you tube video  no man can any videos related to document based cahtbot.....can u make video document base chatbot using lstm,......when u search chatbot everyone focus on intents ,patterns ,response based chatbot predefinedrasa nlu....no make make document based chatbot.........can u make it.........?"]

a=clf.predict(a1).tolist()
for i in range(len(a)):
  if a[i] == 1:
    print("It is a positive comment \U0001f600")
  elif a[i] == 0:
    print("It is a neutral comment \U0001F610")
  else:
    print("It is a negative comment \U0001F614")

It is a neutral comment 😐
It is a neutral comment 😐
It is a negative comment 😔


**Train_test_split-3**

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3,random_state=2022,
    stratify=y)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report
#1. create a pipeline object
clf = Pipeline([
     ('vectorizer_tfidf',TfidfVectorizer()),
     ('Random Forest', RandomForestClassifier())
])

#2. fit with X_train and y_train
clf.fit(X_train, y_train)


#3. get the predictions for X_test and store it in y_pred
y_pred = clf.predict(X_test)


#4. print the confusion matrix and classfication report.
from sklearn.metrics import classification_report, confusion_matrix
print(confusion_matrix(y_test, y_pred))

print()
print(classification_report(y_test, y_pred))

[[  5   3  13]
 [  0  42  28]
 [  1  14 183]]

              precision    recall  f1-score   support

          -1       0.83      0.24      0.37        21
           0       0.71      0.60      0.65        70
           1       0.82      0.92      0.87       198

    accuracy                           0.80       289
   macro avg       0.79      0.59      0.63       289
weighted avg       0.79      0.80      0.78       289



In [ ]:
a1=["worst nlp course ever","which playlist i should follow to get undrestand and build nlp model step by step . i am completely beginner in AI","in any you tube video  no man can any videos related to document based cahtbot.....can u make video document base chatbot using lstm,......when u search chatbot everyone focus on intents ,patterns ,response based chatbot predefinedrasa nlu....no make make document based chatbot.........can u make it.........?"]

a=clf.predict(a1).tolist()
for i in range(len(a)):
  if a[i] == 1:
    print("It is a positive comment \U0001f600")
  elif a[i] == 0:
    print("It is a neutral comment \U0001F610")
  else:
    print("It is a negative comment \U0001F614")

It is a neutral comment 😐
It is a neutral comment 😐
It is a negative comment 😔


**Percentage Calculator**

In [ ]:
def percentage(a,b):
  print(round((a/b)*100,1),"%")

In [ ]:
# Percentage calculator for all the comments.

# Positive percentage
print("The number of positive comments are")
percentage(clf.predict(lb['Comment']).tolist().count(1),len(lb['Comment']))

# Neutral percentage
print("The number of neutral comments are")
percentage(clf.predict(lb['Comment']).tolist().count(0),len(lb['Comment']))

# Negative percentage
print("The number of negative comments are")
percentage(clf.predict(lb['Comment']).tolist().count(-1),len(lb['Comment']))

The number of positive comments are
69.9 %
The number of neutral comments are
24.0 %
The number of negative comments are
6.1 %


**Search function**

In [ ]:
lst = []
time_stamps = []
all = []
for i in lb['Comment']:
  lst.append(i)
i = int(input("What types of comments you want to fetch ?"))

# For timestamps
if(i==1):
  for i in range(len(lst)):
    pattern= '\d{2}:\d{2}'
    matches = re.search(pattern,lst[i])
    if matches:
      print(cm['Name'][i],"||",lst[i])
      time_stamps.append(lst[i])

# For topics extraction
elif(i==2):
  pattern= input()
  for i in range(len(lst)):
    matches = re.search(pattern,lst[i])
    if matches:
      print(cm['Name'][i], "||" ,lst[i])
      all.append(lst[i])


What types of comments you want to fetch ?2
backend
Naushath Raja Mohammed || backend testing is failing in my case: error 405 is coming what should i do @codebasics
soumya ranjan patnaik || ​@codebasics server is running and backend is same as yours. I think there is problem with dialogflow itself....
Because, I tested the webhook in postman and working fine. So it clearly means that dialogflow is unable to make post request on our endpoint.


In [ ]:
# Percentage calculator for specific comments.

# Percentage calculator
print(clf.predict(all))

# Positive percentage
print("The number of positive comments are")
percentage(clf.predict(all).tolist().count(1),len(all))

# Neutral percentage
print("The number of neutral comments are")
percentage(clf.predict(all).tolist().count(0),len(all))

# Negative percentage
print("The number of negative comments are")
percentage(clf.predict(all).tolist().count(-1),len(all))

[-1  1]
The number of positive comments are
50.0 %
The number of neutral comments are
0.0 %
The number of negative comments are
50.0 %


**Now let's pickle that model, which better among the 3 splits.**

In [ ]:
import pickle

In [ ]:
# We will the pickle the model with the train-test split 80-20.
my_model = clf

In [ ]:
with open('my_model.pickle', 'wb') as f:
    pickle.dump(my_model, f)

In [ ]:
with open('my_model.pickle', 'rb') as f:
    loaded_object = pickle.load(f)

In [ ]:
a=["worst nlp course ever","backend testing is failing in my case: error 405 is coming what should i do @codebasics"]
loaded_object.predict(a)

array([ 0, -1])